# Debugging owlready

In [ ]:
import os
import pathlib
import logging
import pandas as pd
from owlready2 import DatatypeProperty, FunctionalProperty, ObjectProperty, AllDisjoint, Thing
from emmopy import get_emmo
from ontopy import World


#import labop_labware_ontology.labop_labware_ontology_impl as lolwi

In [ ]:
# loading EMMO ontology
emmo_world = World()
emmo = emmo_world.get_ontology()
emmo.load()               # reload_if_newer = True
emmo.sync_python_names()  # synchronize annotations
#emmo.base_iri = emmo.base_iri.rstrip('/#')
#self.catalog_mappings = {emmo.base_iri: emmo_url}

In [ ]:
#from owlready2 import *

In [ ]:
# create t-box
lolwt_base_iri = "http://www.labop.org/labop_labware_tbox#"

lolwt = emmo_world.get_ontology(lolwt_base_iri)
lolwt.imported_ontologies.append(emmo)

with lolwt:
    class WellVolume(emmo.Volume):
          """Total Labware volume """
          #volume = emmo.hasQuantityValue.some(emmo.PhysicalQuantity)
          volume = 0.0

    class Labware(emmo.Device):
        """Labware class"""
       
    class hasWellVolume(Labware >> float, FunctionalProperty, DatatypeProperty):
                """Total volume of a Labware """

    class Labware(emmo.Device):
        """Labware class"""
       # is_a = [emmo.Device, hasWellVolume.some(float)]
        


In [ ]:
lolwt.save('labop_labware_tbox_sm.ttl', format='turtle')

## creating A-Box 

In [ ]:
emmo_world = World()
emmo = emmo_world.get_ontology()
emmo.load()               # reload_if_newer = True
emmo.sync_python_names()

In [ ]:
# create t-box
lolwa_base_iri = "http://www.labop.org/labop_labware_abox#"

lolwt_filename = "labop_labware_tbox_sm.ttl"

lolwt = emmo_world.get_ontology(lolwt_filename).load()

lolwa = emmo_world.get_ontology(lolwa_base_iri)
lolwa.imported_ontologies.append(emmo)
lolwa.imported_ontologies.append(lolwt)

## creating individuals


In [ ]:
lolwt.sync_python_names()

In [ ]:
with lolwt:
    mtp_384 = lolwt.Labware("mtp_384", hasWellVolume = 1000)

In [ ]:
lolwt.save('labop_labware_abox_sm.ttl', format='turtle')

In [ ]:
# create a-box

lolwa_base_iri = "http://www.labop.org/labop_labware_abox#"

lolwa = emmo_world.get_ontology(lolwa_base_iri)
lolwa.imported_ontologies.append(emmo)
lolwa.imported_ontologies.append(lolwt)

#lolwt_ns = lolwa.get_namespace(lolwt_base_iri)
with lolwa:
    mtp_384 = Labware("mtp_384", hasWellVolume = 1000)

    

In [ ]:
lolwa.save('labop_labware_abox_sm.ttl', format='turtle')

## Loading Ontologies from files

In [ ]:
#base_iri = 'http://www.labop.org/labop_labware_abox'
#lolwa_version_iri = f'http://www.labop.org/{__version__}/labware-a'

lwemmo_filename = "../ontologies/labop_labware_emmo_ext.ttl"

emmo_world = World()

print("loading ontology from file: ", lwemmo_filename, " ...")
emmo_ext = emmo_world.get_ontology(lwemmo_filename)
#lolwa.base_iri = base_iri
emmo_ext.load()
#emmo_ext.imported_ontologies.append(emmo_ext)
print(" ##### classes:", list(emmo_ext.classes()))  

In [ ]:
emmo_world.ontologies

In [ ]:
emmo = emmo_world.ontologies['http://emmo.info/emmo-inferred#']
emmo.sync_python_names()
emmo.sync_attributes(name_policy="uuid", name_prefix="EMMO_")

In [ ]:
list(emmo.classes())

In [ ]:
def my_render(entity):
    return "%s:%s" % (entity.name, entity.preflabel.first())
set_render_func(my_render)

In [ ]:
list(emmo.classes())

In [ ]:
tbox = emmo_world.ontologies['http://www.labop.org/labop_labware_tbox#']
list(tbox.classes())

In [ ]:
#base_iri = 'http://www.labop.org/labop_labware_abox'
#lolwa_version_iri = f'http://www.labop.org/{__version__}/labware-a'

lw_tbox_filename = "../ontologies/labop_labware_tbox.ttl"

print("loading ontology from file: ", lw_tbox_filename, " ...")
lolwt = emmo_world.get_ontology(lw_tbox_filename)
#lolwa.base_iri = base_iri
lolwt.load()
emmo_ext.imported_ontologies.append(lolwt)

print(" ##### classes:", list(lolwt.classes()))  

In [ ]:
emmo_world.ontologies

In [ ]:
lolwt.Labware.iri

In [ ]:
lolwt.base_iri

In [ ]:
lolwt.Device.iri

In [ ]:
lw_abox_filename = "../ontologies/labop_labware_abox.ttl"

print("loading ontology from file: ", lw_abox_filename, " ...")
lolwa = emmo_world.get_ontology(lw_abox_filename)
#lolwa.base_iri = base_iri
lolwa.load()
#emmo_ext.imported_ontologies.append(lolwa)

print(" ##### classes:", list(lolwa.classes()))  

In [ ]:
lolwa.base_iri

In [ ]:
print(" ##### classes:", list(lolwt.classes()))  

In [ ]:
def clean_id(id_str):
    return id_str.replace("-","_").replace(" ","").strip().lower()

In [ ]:
import pandas as pd
import numpy as np

lolwt_filename = "labop_labware_tbox.ttl"
csv_filename="../labware_catalogues/labop_labware_catalog_medium.csv"

emmo_world = World()

print("loading ontology from file: ", lolwt_filename, " ...")
lolwt = emmo_world.get_ontology(lolwt_filename).load()

lolwa = emmo_world.get_ontology("http://www.labop.org/labop_labware_abox#")

lolwa.imported_ontologies.append(lolwt)


labware_cat_df = pd.read_csv(csv_filename, delimiter=";")
labware_cat_df = labware_cat_df.reset_index()  # make sure indexes pair with number of rows


labware_cat_df

In [ ]:
manufacturer_dict = {}
with lolwa:

     for index,row in labware_cat_df.iterrows():
                #print( self.clean_id(row['Id']), "-- >", row['Manufacturer'], row['ProductID'], row['UNSPSC'], "EC: ", row['eClass'] )
                #print( self.clean_id(row['Id']), "-- >", row['Manufacturer'], row['ProductID'], row['WellCount'] )

                # create the manufacturer
                if row['Manufacturer'] not in manufacturer_dict:
                    curr_manufacturer = lolwt.Manufacturer(row['Manufacturer'])
                    #curr_manufacturer.hasName(primaryName=row['Manufacturer'])
                    manufacturer_dict[row['Manufacturer']] = curr_manufacturer

                law = lolwt.Labware( clean_id(row['Id']),
                                        # ;Description;ImageLink/URL;UNSPSC;eClass;Vendor;CatalogueID;WellCount;ColumnCount;RowCount;LabwareLength/mm;LabwareWidth/mm;LabwareHeight/mm;Mass/g;LabwareMaterial;SurfaceTreatment;Color;WellVolume/ul;A1Position(col,row);WellDiameter/mm;WellColDistance/mm;WellRowDistance/mm;WellDepth/mm;WellShape;WellBottomShape;Liddable/bool;Lid((Manufacturer, ProdID));Applications;AcceptableLids
                                        hasManufacturer=manufacturer_dict[row['Manufacturer']],
                                        hasProductID=row['ProductID'] if row['ProductID'] is not np.nan else "unknown",
                                        # LabWareType
                                        # Description
                                        #hasImageLink=row['ImageLink/URL'] if row['ImageLink/URL'] is not np.nan else "http://",
                                        #hasUNSPSC=row['UNSPSC'] if row['UNSPSC'] is not np.nan else "unknown",
                                        #hasEClass=row['eClass'] if row['eClass'] is not np.nan else "unknown",
                                        hasVendorName=row['Vendor'],
                                        #hasVendorProductID=row['CatalogueNumber'],
                                        hasNumWells=int(row['WellCount']) if row['WellCount'] is not np.nan else None, # TODO: check if this is correct
                                        hasNumCols=int(row['ColumnCount']) if row['ColumnCount'] is not np.nan else 0, # TODO: should be None
                                        hasNumRows=int(row['RowCount']) if row['RowCount'] is not np.nan else 0,)

     


In [ ]:
lolwa.save("lolwa_individuals.ttl", format="turtle")

In [ ]:
prefix_dict = {
    'rdf': "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    'rdfs': "http://www.w3.org/2000/01/rdf-schema#",
    'xml': "http://www.w3.org/XML/1998/namespace",
    'xsd': "http://www.w3.org/2001/XMLSchema#",
    'owl': "http://www.w3.org/2002/07/owl#",
    'skos': "http://www.w3.org/2004/02/skos/core#",
    'dc': "http://purl.org/dc/elements/1.1/",
    'dcterm': "http://purl.org/dc/terms/",
    'dctype': "http://purl.org/dc/dcmitype/",
    'foaf': "http://xmlns.com/foaf/0.1/",
    'wd': "http://www.wikidata.org/entity/",
    'ex': "http://www.example.com/",
    'emmo': "http://emmo.info/emmo#",
    'lolw': "http://www.labop.org/labware'#",
}

In [ ]:

graph = emmo_world.as_rdflib_graph()

for prefix, iri in prefix_dict.items():
    print(prefix, "--- ", iri )
    graph.bind(prefix, iri)

In [ ]:
query = """

PREFIX lolwa: <http://www.labop.org/labware#>

SELECT ?lw_name ?lw_num_wells 
WHERE {
    ?lw_name rdf:type lolw:Labware.
    ?lw_name lolw:hasManifacturer "Greiner".
    ?lw_name lolw:hasNumWells ?lw_num_wells.
    }
"""

## Example from owlready documentation

In [ ]:
import csv
import owlready2

In [ ]:
onto = owlready2.get_ontology("bacteria.owl").load()

In [ ]:
onto_individuals = owlready2.get_ontology("http://lesfleursdunormal.fr/static/_downloads/bacteria_individuals.owl")

In [ ]:
onto_individuals.imported_ontologies.append(onto)

In [ ]:
with onto_individuals:
    individual = onto.Bacterium("bac1")

    individual.gram_positive = True

    individual.nb_colonies = 33



In [ ]:
onto_individuals.save("bacteria_individuals.owl")

## now the same with EMMO

In [ ]:
lwemmo_filename = "bacteria.owl"

emmo_world = World()

print("loading ontology from file: ", lwemmo_filename, " ...")
emmo_ext = emmo_world.get_ontology(lwemmo_filename).load()

emmo_individuals = emmo_world.get_ontology("http://www.labop.org/labop_labware_abox#")

emmo_individuals.imported_ontologies.append(emmo_ext)

with emmo_individuals:
    individual = emmo_ext.Bacterium("bac1")

    individual.gram_positive = False    

    individual.nb_colonies = 42

emmo_individuals.save("bacteria_individuals.ttl", format="turtle")



In [ ]:
lwemmo_filename = "labop_labware_tbox_sm.ttl"

emmo_world = World()

print("loading ontology from file: ", lwemmo_filename, " ...")
emmo_ext = emmo_world.get_ontology(lwemmo_filename).load()

emmo_individuals = emmo_world.get_ontology("http://www.labop.org/labop_labware_abox#")

emmo_individuals.imported_ontologies.append(emmo_ext)

with emmo_individuals:
    individual = emmo_ext.Labware("lw1")


    individual.hasWellVolume = 42

emmo_individuals.save("lw_individuals.ttl", format="turtle")